# Phase 3B Preprocessing: Targeted Signal Investigation

This notebook executes the **Targeted Signal Investigation** (`imagined_speech.data.targeted_investigation`) on the three extreme cases identified during the Phase 3B quality review.

### Target Investigation Cases
1. **Case 1: Extreme Amplitude** — `Digit/name6_6.edf`, channel **O2** (post-filter PTP $\approx 7,795.86\ \mu\text{V}$).
2. **Case 2: High Boundary-to-Interior Ratio** — `Image/name16_Mobile.edf`, channel **P8** (boundary/interior std ratio $\approx 13.77\times$).
3. **Case 3: Extremely Low Standard Deviation** — `Char/name14_S.edf`, channel **T8** (post-filter std $\approx 0.2921\ \mu\text{V} < 0.5\ \mu\text{V}$ threshold).

### Scientific Safeguards & Rigorous Investigation Principles
- **Read-Only Invariant**: Raw EDF files, Phase 1–3B artifacts, and configuration files are loaded strictly read-only. No input files are modified.
- **Standardized Units & Time Axes**: All signal amplitudes are evaluated and visualized in native microvolts ($\mu\text{V}$); time axes are plotted in elapsed seconds ($s$) at $128.0\ \text{Hz}$.
- **Sensor Dynamic Range & Calibration Audit**: Physical and digital ranges, dimensions, and LSB resolution ($\sim 0.502\ \mu\text{V/count}$ for Emotiv EPOC 16-bit ADC) are extracted directly from EDF headers to determine if extreme amplitudes are empirical events present in the raw data or artifacts of digital processing.
- **Excursion Timing & Padding Evaluation**: The timing and magnitude of boundary excursions are quantified directly in raw and filtered traces without assuming filter-padding failure.
- **Quantization vs. Attenuation Analysis**: Distinguishes true flatlines, discrete step quantization, and low-amplitude continuous physiological activity for traces flagged by exploratory low-variance screening.
- **Strict Checksum Integrity**: Verifies byte-for-byte SHA-256 checksums before and after the investigation against pre-review snapshots or the original ZIP archive (`Imagined_speech_EEG_edf.zip`). If any baseline snapshot is unavailable, reports that limitation explicitly rather than claiming verified integrity.
- **100% Data Retention**: No recordings or channels are rejected, interpolated, or normalized; CAR remains non-default; no windowing, train/test splitting, or model training is introduced.

## 1. Mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Google Colab: {IN_COLAB}')

## 2. Clone or Update the Repository

In [ ]:
import subprocess

REPO_URL = 'https://github.com/ACT2039/imagined-speech-eeg.git'
COLAB_REPO_DIR = Path('/content/imagined-speech-eeg')

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        print(f'Cloning {REPO_URL} into {COLAB_REPO_DIR}...')
        subprocess.run(['git', 'clone', REPO_URL, str(COLAB_REPO_DIR)], check=True)
    elif (COLAB_REPO_DIR / '.git').is_dir():
        print(f'Updating existing repository at {COLAB_REPO_DIR}...')
        subprocess.run(['git', '-C', str(COLAB_REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        raise RuntimeError(
            f'Directory {COLAB_REPO_DIR} exists but is not a valid Git repository.'
        )
    os.chdir(COLAB_REPO_DIR)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        f'pyproject.toml not found in working directory {REPO_ROOT}. '
        'Ensure the notebook is running from the repository root.'
    )

print(f'Repository root: {REPO_ROOT}')

## 3. Install Package and Verify Dependencies

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

src_dir = str((REPO_ROOT / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import imagined_speech
print(f'Installed imagined_speech version: {imagined_speech.__version__}')

## 4. Resolve `DATA_ROOT`, Phase 3B Preprocessing Directory, and Checksum Source

In [ ]:
import json
import zipfile
import pandas as pd
from IPython.display import display

# Common dataset paths in Google Drive and Colab local disk
CONFIRMED_COLAB_DATA_ROOT = Path('/content/data/raw/Imagined_speech_EEG_edf')
DRIVE_ZIP_PATH = Path(
    os.environ.get(
        'ISR_DRIVE_ZIP',
        '/content/drive/MyDrive/datasets/Imagined_speech_EEG_edf.zip',
    )
)
ZIP_CANDIDATE_PATHS = [
    DRIVE_ZIP_PATH,
    Path('/content/drive/MyDrive/Imagined_speech_EEG_edf.zip'),
    Path('/content/data/raw/Imagined_speech_EEG_edf.zip'),
    Path('./data/raw/Imagined_speech_EEG_edf.zip'),
]

# Resolve original ZIP archive if present
ORIGINAL_ZIP_PATH = None
for zp in ZIP_CANDIDATE_PATHS:
    if zp and zp.is_file():
        ORIGINAL_ZIP_PATH = zp.resolve()
        break

if os.environ.get('ISR_DATA_ROOT'):
    DATA_ROOT = Path(os.environ['ISR_DATA_ROOT']).resolve()
elif IN_COLAB and CONFIRMED_COLAB_DATA_ROOT.is_dir():
    DATA_ROOT = CONFIRMED_COLAB_DATA_ROOT.resolve()
else:
    DATA_ROOT = Path('/content/data/raw' if IN_COLAB else './data/raw').resolve()

if ORIGINAL_ZIP_PATH is None:
    print(
        f'[LIMITATION WARNING] Original dataset ZIP archive was NOT found at {DRIVE_ZIP_PATH} '
        'or alternative candidate paths.\n'
        '  - Direct byte-level comparison against original ZIP members will be UNAVAILABLE.\n'
        '  - Raw EDF files can only be checked against pre-review snapshots if available.\n'
        '  - Integrity verification will explicitly report this limitation rather than claiming verified status.'
    )
    if not any(DATA_ROOT.rglob('*.edf')):
        raise FileNotFoundError(
            f'Raw EDF files not found under {DATA_ROOT} and dataset archive is missing at {DRIVE_ZIP_PATH}. '
            'Please mount Google Drive containing the dataset archive.'
        )
else:
    print(f'[FOUND ZIP] Original archive located at: {ORIGINAL_ZIP_PATH}')

# Auto-extract ZIP if Colab runtime was recycled and DATA_ROOT is empty
if ORIGINAL_ZIP_PATH and not any(DATA_ROOT.rglob('*.edf')):
    print(f'Extracting raw EDF files from {ORIGINAL_ZIP_PATH} to {DATA_ROOT}...')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ORIGINAL_ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_ROOT.parent if DATA_ROOT.name == 'Imagined_speech_EEG_edf' else DATA_ROOT)

PHASE3B_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_PHASE3B_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase3b_preprocessing'
        if IN_COLAB
        else './outputs/phase3b_preprocessing',
    )
)

INVESTIGATION_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_INVESTIGATION_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase3b_targeted_investigation'
        if IN_COLAB
        else './outputs/phase3b_targeted_investigation',
    )
)

print(f'Resolved DATA_ROOT:                 {DATA_ROOT}')
print(f'Resolved Original ZIP:              {ORIGINAL_ZIP_PATH}')
print(f'Resolved Phase 3B Preprocessing:    {PHASE3B_OUTPUT_DIR}')
print(f'Target Investigation Directory:     {INVESTIGATION_OUTPUT_DIR}')

## 5. Compute Pre-Investigation SHA-256 Baseline Snapshots
Computes byte-level SHA-256 hashes of input files before the investigation begins. If any directory is unavailable or empty, explicitly notes that limitation.

In [ ]:
from imagined_speech.data.preprocessing_review import compute_directory_sha256_snapshot
from imagined_speech.data.targeted_investigation import compute_zip_edf_sha256_snapshot

print('Computing pre-investigation SHA-256 snapshots...')
if ORIGINAL_ZIP_PATH and ORIGINAL_ZIP_PATH.is_file():
    print(f'Building raw EDF baseline snapshot from original ZIP archive: {ORIGINAL_ZIP_PATH.name}...')
    raw_edf_hashes_before = compute_zip_edf_sha256_snapshot(ORIGINAL_ZIP_PATH)
    if raw_edf_hashes_before:
        print(f'[BASELINE AVAILABLE] raw_edfs: {len(raw_edf_hashes_before)} files snapshotted from ZIP via SHA-256.')
    else:
        print('[LIMITATION] Failed to build baseline snapshot from ZIP archive.')
else:
    raw_edf_hashes_before = compute_directory_sha256_snapshot(DATA_ROOT, pattern='*.edf')
    if raw_edf_hashes_before:
        print(f'[BASELINE AVAILABLE] raw_edfs: {len(raw_edf_hashes_before)} files snapshotted from {DATA_ROOT} via SHA-256.')
    else:
        print('[LIMITATION] Baseline snapshot for raw_edfs is unavailable.')

phase3b_hashes_before = compute_directory_sha256_snapshot(PHASE3B_OUTPUT_DIR)

baseline_snapshots = {
    'raw_edfs': raw_edf_hashes_before,
    'phase3b_preprocessing': phase3b_hashes_before,
}

for stage_name, hashes in baseline_snapshots.items():
    if hashes:
        print(f'[BASELINE AVAILABLE] {stage_name}: {len(hashes)} files snapshotted via SHA-256.')
    else:
        print(f'[LIMITATION] Baseline snapshot for {stage_name} is unavailable. Full byte-level integrity verification cannot be performed for this stage.')

## 6. Run Targeted Investigation Unit Tests

In [ ]:
print('Running unit tests for targeted signal investigation...')
pytest_proc = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_targeted_investigation.py', '-v'],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    check=False,
)
if pytest_proc.stdout:
    print(pytest_proc.stdout)
if pytest_proc.stderr:
    print(pytest_proc.stderr, file=sys.stderr)
if pytest_proc.returncode != 0:
    raise RuntimeError(
        f'Targeted investigation unit tests failed with exit code {pytest_proc.returncode}. '
        'See captured pytest output above.'
    )
print('Targeted investigation unit tests passed successfully!')

## 7. Targeted Investigation Smoke Test
Executes a smoke run of `run_phase3b_targeted_investigation` in an isolated temporary directory using synthetic EDF/NPZ files to ensure flawless end-to-end orchestration before investigating real recordings.

In [ ]:
import tempfile
import numpy as np
from imagined_speech.config import DEFAULT_PRIMARY_EEG_MONTAGE
from imagined_speech.data.preprocessing import save_preprocessed_recording_npz
from imagined_speech.data.targeted_investigation import (
    TargetInvestigationCase,
    run_phase3b_targeted_investigation,
)
from tests.conftest import write_synthetic_edf

with tempfile.TemporaryDirectory(prefix='targeted_smoke_') as smoke_tmp:
    smoke_root = Path(smoke_tmp)
    smoke_raw = smoke_root / 'raw'
    smoke_p3b = smoke_root / 'phase3b'
    smoke_out = smoke_root / 'investigation_out'

    # Generate synthetic EDF and NPZ for a test case
    smoke_edf = smoke_raw / 'Digit' / 'name6_6.edf'
    sfreq = 128.0
    n_samples = 128 * 12
    ch_names = list(DEFAULT_PRIMARY_EEG_MONTAGE)
    signals = [np.sin(np.linspace(0, 10, n_samples)) * 50.0 + 4000.0 for _ in ch_names]
    write_synthetic_edf(smoke_edf, ch_names, signals, sfreq=sfreq)

    filt_data = np.zeros((len(ch_names), n_samples), dtype=np.float32)
    save_preprocessed_recording_npz(
        output_path=smoke_p3b / 'signals' / 'Digit' / 'name6_6.npz',
        preprocessed_data=filt_data,
        channel_names=ch_names,
        sampling_frequency=sfreq,
        relative_source_path='Digit/name6_6.edf',
        filter_params={'bandpass_low_hz': 0.5, 'bandpass_high_hz': 45.0, 'order': 4},
        applied_reference_mode='none',
    )

    smoke_case = TargetInvestigationCase(
        case_id='smoke_case',
        relative_source_path='Digit/name6_6.edf',
        channel_name='O2',
        issue_description='Smoke test case',
        target_metric='smoke_ptp',
        observed_metric_value=50.0,
    )

    smoke_report = run_phase3b_targeted_investigation(
        data_root=smoke_raw,
        preprocessed_dir=smoke_p3b,
        output_dir=smoke_out,
        cases=[smoke_case],
        save_outputs=True,
        generate_figures=True,
    )
    assert len(smoke_report.case_results) == 1
    assert (smoke_out / 'targeted_trace_statistics.csv').is_file()
    assert (smoke_out / 'phase3b_targeted_investigation_report.md').is_file()
    print('Targeted investigation smoke test passed and validated successfully!')

## 8. Execute Targeted Signal Investigation on Target Cases
Loads the original EDF files and Phase 3B `.npz` files read-only, analyzes calibration metadata, evaluates boundary excursion dynamics, assesses quantization vs. attenuation, and produces high-resolution comparative figures.

In [ ]:
from imagined_speech.data.targeted_investigation import run_phase3b_targeted_investigation

investigation_report = run_phase3b_targeted_investigation(
    data_root=DATA_ROOT,
    preprocessed_dir=PHASE3B_OUTPUT_DIR,
    output_dir=INVESTIGATION_OUTPUT_DIR,
    zip_path=ORIGINAL_ZIP_PATH,
    baseline_snapshots=baseline_snapshots,
    save_outputs=True,
    generate_figures=True,
)

summary_json_path = INVESTIGATION_OUTPUT_DIR / 'phase3b_targeted_investigation_summary.json'
if summary_json_path.is_file():
    print('=== Phase 3B Targeted Investigation Summary ===')
    print(summary_json_path.read_text(encoding='utf-8'))

zip_ver = investigation_report.zip_verification
print('\n=== Raw EDF Baseline Integrity Check Result ===')
print(f"Verification Status: {zip_ver.get('status')}")
print(f"Comparison Source:   {zip_ver.get('comparison_source')}")
print(f"Detail:              {zip_ver.get('detail')}")
if zip_ver.get('status') == 'verified_against_zip_archive':
    print(f"[VERIFIED] Raw EDFs verified byte-for-byte against original ZIP archive.")
elif zip_ver.get('status') == 'baseline_snapshot_unavailable':
    print('[LIMITATION] Baseline archive/snapshot was unavailable; comparison is UNVERIFIED (not passed).')
elif zip_ver.get('status') == 'integrity_violation':
    raise RuntimeError(f"Integrity violation: {zip_ver.get('detail')}")

## 9. Comparative Trace Statistics Table

In [ ]:
print('=== Comparative Raw vs. Preprocessed Trace Statistics ===')
display(
    investigation_report.statistics_df[
        [
            'case_id',
            'relative_source_path',
            'channel_name',
            'raw_mean_uV',
            'raw_std_uV',
            'raw_ptp_uV',
            'filtered_mean_uV',
            'filtered_std_uV',
            'filtered_ptp_uV',
            'physical_min_uV',
            'physical_max_uV',
            'lsb_resolution_uV_per_count',
        ]
    ]
)

## 10. Multi-Panel Diagnostic Visualizations
Displays comparative 3-panel figures for each investigated case, with all amplitudes in native microvolts ($\mu\text{V}$) and time in elapsed seconds ($s$).

In [ ]:
from IPython.display import Image

figs_dir = INVESTIGATION_OUTPUT_DIR / 'figures'
fig_files = sorted(figs_dir.glob('*.png'))
if fig_files:
    for fig_path in fig_files:
        print(f'=== Figure: {fig_path.name} ===')
        display(Image(filename=str(fig_path)))
else:
    print(f'No figures found in {figs_dir}.')

## 11. Targeted Investigation Markdown Report

In [ ]:
from IPython.display import Markdown

report_md_path = INVESTIGATION_OUTPUT_DIR / 'phase3b_targeted_investigation_report.md'
if report_md_path.is_file():
    display(Markdown(report_md_path.read_text(encoding='utf-8')))
else:
    print(f'Report markdown not found at {report_md_path}.')

## 12. Post-Investigation SHA-256 Checksum Audit
Verifies that raw EDF files and Phase 3B preprocessed outputs were not modified during the investigation. Verifies against the original ZIP archive or reports limitations if snapshots are unavailable.

In [ ]:
from imagined_speech.data.preprocessing_review import verify_directory_checksums

print('=== Post-Investigation SHA-256 Checksum Audit ===')
stages_to_check = [
    ('raw_edfs', DATA_ROOT, baseline_snapshots.get('raw_edfs'), '*.edf'),
    ('phase3b_preprocessing', PHASE3B_OUTPUT_DIR, baseline_snapshots.get('phase3b_preprocessing'), '*'),
]

for stage_name, stage_dir, baseline_dict, pat in stages_to_check:
    ver_res = verify_directory_checksums(
        baseline_hashes=baseline_dict,
        current_directory=stage_dir,
        stage_name=stage_name,
        pattern=pat,
    )
    if ver_res['status'] == 'verified_unmodified':
        print(f"[VERIFIED] {stage_name}: All {ver_res['n_files_checked']} files verified byte-for-byte identical via SHA-256.")
    elif ver_res['status'] == 'baseline_snapshot_unavailable':
        print(f"[LIMITATION] {stage_name}: {ver_res['detail']}")
    else:
        raise RuntimeError(f"Integrity violation in {stage_name}: {ver_res['detail']}")

print('\n=== Summary of Baseline Integrity Verification ===')
if zip_ver.get('status') == 'verified_against_zip_archive':
    print(f"[VERIFIED] Raw EDF Baseline: {zip_ver.get('detail')}")
elif zip_ver.get('status') == 'verified_against_snapshot':
    print(f"[VERIFIED] Raw EDF Baseline: {zip_ver.get('detail')}")
elif zip_ver.get('status') == 'baseline_snapshot_unavailable':
    print(f"[LIMITATION] Raw EDF Baseline: {zip_ver.get('detail')}")
elif zip_ver.get('status') == 'integrity_violation':
    raise RuntimeError(f"Raw EDF Baseline integrity violation: {zip_ver.get('detail')}")

print('\nTargeted investigation completed with 100% data retention and integrity verification!')